# Student Group Formation Algorithm

This notebook implements an intelligent student grouping system that balances similarity and diversity to form balanced groups for Architecture Ateliers.

## Overview
The algorithm:
1. **Computes similarity scores** based on shared preferences.
2. **Computes diversity scores** based on different characteristics.
3. **Selects group leaders** based on students preferenced role.

## Requirements
- pandas
- numpy
- random

## Input
- CSV file with student survey responses.
- Must include an 'ID' column for student number.

In [ ]:
import numpy as np
import pandas as pd
import random

# ===== Configuration =====
file_name = ".csv"
index_column = 'ID'
role_question = "Q14"
leader_answer = "Taking charge and organizing tasks (group leader)"
software_question = "Q11"
max_group_size = 15
group_column = "Group_number"
output_csv = ".csv"

# Questions to consider for similarity matching (students with same answers should be together)
SIMILARITY_QUESTIONS = {5, 6, 7, 8, 9, 10, 15, 16}

# Questions to consider for diversity (students with different answers should be together)
DIVERSITY_QUESTIONS = {3, 4, 11, 12, 13, 14}

In [ ]:
df = pd.read_csv(file_name, sep=';')
df.set_index(index_column, inplace=True)
mapper_columns = {}
for i,j in enumerate(df.columns):
    mapper_columns[j] = 'Q'+str(i)
df.rename(columns=mapper_columns, inplace=True)

In [ ]:
# Compute similarity matrix
def compute_similarity(df, similarity_questions):
    n = len(df)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                similarity_matrix[i, j] = 1  # Max similarity with itself
            else:
                score = 0
                for q in similarity_questions:
                    score += (df.iloc[i][q] == df.iloc[j][q])  # Exact match = 1, otherwise 0
                
                similarity_matrix[i, j] = score / len(similarity_questions)  # Normalize

    return pd.DataFrame(similarity_matrix, index=df.index, columns=df.index)

# Compute diversity matrix
def compute_diversity(df, diversity_questions):
    n = len(df)
    diversity_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                diversity_matrix[i, j] = 0  # No diversity with oneself
            else:
                score = 0
                for q in diversity_questions:
                    
                    if isinstance(df.iloc[i][q], set):
                        unique = len(df.iloc[i][q] ^ df.iloc[j][q])
                        total = len(df.iloc[i][q] | df.iloc[j][q])
                        score += unique / total if total > 0 else 0
                    
                    elif isinstance(df.iloc[i][q], int):
                        max_range = 5  # rating is 1-5
                        score += 1 - abs(df.iloc[i][q] - df.iloc[j][q]) / max_range
                    
                    else:
                        score += (df.iloc[i][q] != df.iloc[j][q])
                
                diversity_matrix[i, j] = score / len(diversity_questions)  # Normalize

    return pd.DataFrame(diversity_matrix, index=df.index, columns=df.index)

# select 3 starting students
def select_leaders(df, similarity_df, diversity_df):
    leader_candidates = df[df[role_question] == "Taking charge and organizing tasks (group leader)"].index.tolist()

    # Step 1: Randomly pick first leader
    first_leader = random.choice(leader_candidates)

    # Step 2: Pick the second leader with the highest diversity score
    remaining_candidates = [c for c in leader_candidates if c != first_leader]
    if remaining_candidates:
        second_leader = max(remaining_candidates, key=lambda x: diversity_df.loc[first_leader, x])
    else:
        second_leader = None

    # Step 3: Pick the third leader with the lowest similarity to the first two
    remaining_candidates = [c for c in leader_candidates if c not in [first_leader, second_leader]]
    if remaining_candidates:
        third_leader = min(remaining_candidates, key=lambda x: similarity_df.loc[first_leader, x])
    else:
        third_leader = None

    # Return only valid leaders
    return [leader for leader in [first_leader, second_leader, third_leader] if leader is not None]

# Assign students to groups using average affinity to current group members (greedy)
def assign_groups(affinity_df, initial_members, group_size=None):
    # Initialize groups
    groups = {f"Group_{i+1}": [m] for i, m in enumerate(initial_members)}

    all_students = set(affinity_df.index)
    unassigned = list(all_students - set(initial_members))

    # Helper: compute average affinity of a student to a group
    def avg_affinity_to_group(student, members):
        return float(np.mean([affinity_df.loc[student, m] for m in members]))

    while unassigned:
        best_student = None
        best_group = None
        best_score = -1e9

        # Pick the best (student, group) move available (greedy)
        for student in unassigned:
            for gname, members in groups.items():
                if group_size is not None and len(members) >= group_size:
                    continue
                score = avg_affinity_to_group(student, members)

                # Tie-break: prefer smaller group
                if (score > best_score) or (score == best_score and best_group is not None and len(groups[gname]) < len(groups[best_group])):
                    best_score = score
                    best_student = student
                    best_group = gname

        # Fallback safety (should rarely happen unless all groups are full)
        if best_student is None:
            # Put remaining students into any group with space
            for student in unassigned:
                for gname, members in groups.items():
                    if group_size is None or len(members) < group_size:
                        groups[gname].append(student)
                        break
            break

        groups[best_group].append(best_student)
        unassigned.remove(best_student)

    return groups

# add groups information to the main df
def integrate_groups(df, groups):
    # Create a mapping of students to their assigned groups
    student_to_group = {}
    for group_name, members in groups.items():
        for student in members:
            student_to_group[student] = group_name

    # Add a new column to the df with the group assignments
    df[group_column] = df.index.map(student_to_group)

    return df

In [ ]:
df_sim = compute_similarity(df, SIMILARITY_QUESTIONS)
df_div = compute_diversity(df, DIVERSITY_QUESTIONS)
affinity = df_sim + df_div

In [ ]:
leaders = select_leaders(df, df_sim, df_div)

# Assign groups
groups = assign_groups(affinity, leaders, max_group_size)

# Display the final groups
print("Final Groups:")
for group_name, members in groups.items():
    print(f"{group_name}: {members}")

df_with_groups = integrate_groups(df,groups)
df_with_groups.to_csv(output_csv)

# Group Composition Analysis

The following cells analyze the distribution of student characteristics across the formed groups to verify balance and diversity.

In [ ]:
# ===== Analysis Functions =====
def counts_by_group(df, group_col, question_col, normalize=False):
    """
    Analyzes answer distributions across groups for a single-select question.
    
    Args:
        df: DataFrame with group and question columns
        group_col: Name of the column containing group assignments
        question_col: Name of the question column to analyze
        normalize: If True, returns percentages; if False, returns counts
        
    Returns:
        DataFrame with rows=groups, columns=answers, values=count or percentage
    """
    tab = pd.crosstab(df[group_col], df[question_col], normalize="index" if normalize else False)
    return tab

def multi_select_counts_by_group(df, group_col, question_col, sep):
    """
    Analyzes answer distributions across groups for a multi-select question.
    Splits comma-separated values and counts each selection separately.
    
    Args:
        df: DataFrame with group and question columns
        group_col: Name of the column containing group assignments
        question_col: Name of the multi-select question column
        sep: Separator character used in the multi-select field (e.g., ",")
        
    Returns:
        DataFrame with rows=groups, columns=answer options, values=counts
    """
    tmp = df[[group_col, question_col]].copy()
    tmp[question_col] = tmp[question_col].fillna("").astype(str)

    exploded = (
        tmp.assign(item=tmp[question_col].str.split(sep))
           .explode("item")
    )
    exploded["item"] = exploded["item"].str.strip()
    exploded = exploded[exploded["item"] != ""]

    return pd.crosstab(exploded[group_col], exploded["item"])

In [ ]:
counts_by_group(df_with_groups, group_column, "Q2", normalize=False)

In [ ]:
# Analyze multi-select software tools/skills distribution across groups
multi_select_counts_by_group(df_with_groups, group_column, software_question, ",")